# **📘 Image Analysis & Clustering Pipeline — Explanation**

Este documento descreve todo o pipeline de análise de imagens, incluindo extração de características, clusterização com dois métodos (KMeans e HDBSCAN), detecção de outliers, análise de similaridade, deduplicação e visualização final.

O objetivo é **entender o dataset**, **diagnosticar seus problemas**, **melhorar qualidade**, e **identificar padrões**, orientando decisões para ML e visão computacional.

---

# **1. Overview of the Pipeline**

O pipeline processa todas as imagens de uma pasta e executa:

1. **Descoberta das imagens**
2. **Extração de metadados**
3. **Extração de embeddings com ResNet50**
4. **Redução de dimensionalidade com PCA**
5. **Clusterização com KMeans**
6. **Clusterização com HDBSCAN**
7. **Detecção de outliers**
8. **Similaridade e deduplicação**
9. **Avaliação automática dos clusterizadores**
10. **Geração de visualizações**  
11. **Heatmap associativo KMeans × HDBSCAN**
12. **Relatório HTML completo**

O resultado final inclui arquivos:

- `clusters_kmeans.png`
- `clusters_hdbscan.png`
- `clusters_overlay.png`
- `cluster_heatmap.png`
- `image_summary.csv`
- `image_metadata.json`
- `similar_images.json`
- `duplicate_images.json`
- `report.html`

---

# **2. Step-by-Step Explanation**

---

## **2.1 Image Discovery**

O sistema varre recursivamente uma pasta, encontrando arquivos:

- jpg  
- jpeg  
- png  
- bmp  
- tiff  

Isso define o conjunto completo de imagens para análise.

---

## **2.2 Metadata Extraction**

Para cada imagem, extraímos:

- largura, altura, formato  
- estatísticas: média, desvio padrão, mínimos/máximos  
- EXIF (quando disponível):  
  – câmera, data, exposição, etc.  

Isso ajuda a identificar:

- imagens muito escuras/clara  
- resoluções inesperadas  
- padrões de captura  
- possíveis erros de aquisição  

---

## **2.3 CNN Embeddings (ResNet50)**

Cada imagem passa por uma ResNet50 pré-treinada (ImageNet).  
O resultado é um vetor de ~2048 dimensões que representa:

- textura  
- forma  
- bordas  
- conteúdo  
- características semânticas aprendidas  

Em outras palavras:

**Imagens parecidas → embeddings parecidos.**

Esses vetores alimentam todos os próximos módulos.

---

## **2.4 PCA (Principal Component Analysis)**

Embeddings de 2048 dimensões são grandes demais.  
PCA reduz para 50 dimensões com:

- preservação da variância  
- remoção de ruído  
- estrutura mais compacta  
- clusters mais estáveis  
- visualização possível em 2D  

PCA é essencial para desempenho e estabilidade.

---

# **3. Clustering Approaches**

---

# **3.1 KMeans Clustering**

### **Como funciona**

KMeans:

1. Escolhe *k* centróides  
2. Atribui cada ponto ao centro mais próximo  
3. Recalcula centróides  
4. Repete até convergir  

### **Propriedades**

- rápido  
- fácil de interpretar  
- assume clusters “esféricos”  
- força todos os itens a pertencerem a algum cluster  

### **Quando funciona bem**

- embeddings limpos de CNN (como ResNet50)  
- clusters relativamente homogêneos  
- dataset com boa separabilidade  

---

# **3.2 HDBSCAN Clustering**

### **Como funciona**

HDBSCAN é baseada em densidade:

- encontra *regiões densas* no espaço  
- não precisa de k  
- clusters podem ter formas arbitrárias  
- pontos fora dessas regiões = **noise (-1)**  

### **Vantagens**

- robusto contra ruído  
- identifica grupos pequenos (que KMeans ignora)  
- clusters com formas complexas  
- não obriga todo mundo a pertencer a um cluster  

### **Quando funciona bem**

- datasets com muita variação  
- estruturas complexas  
- imagens com muitos tipos diferentes  
- presença de outliers naturais  

---

# **4. Outlier Detection**

Além dos algoritmos de clusterização, o pipeline usa:

### **Isolation Forest**

Excelente para detectar:

- imagens completamente diferentes  
- muito escuras/claras  
- perspectiva irregular  
- objetos raros  
- ruído e erros de captura  

Esses outliers ajudam a explicar por que um modelo ML falha.

---

# **5. Similarity & Deduplication**

Com **cosine similarity** sobre os embeddings:

- distância pequena → imagens extremamente parecidas  
- similaridade > 0.90 → imagens quase iguais  
- similaridade > 0.98 → duplicadas  

Exportamos:

- `similar_images.json`  
- `duplicate_images.json`  

Isso é crucial para:

- limpeza do dataset  
- remoção de redundância  
- identificar conjuntos de frames quase idênticos  

---

# **6. Comparison Between KMeans × HDBSCAN**

O pipeline gera:

### **(a) Três gráficos**
- clusters_kmeans.png  
- clusters_hdbscan.png  
- clusters_overlay.png  

### **(b) Heatmap**
- cluster_heatmap.png  
Mostra como clusters de um método mapeiam para o outro.

### **(c) Métricas**
- Silhouette  
- Davies–Bouldin  
- Calinski–Harabasz  

### **(d) Avaliação automática**
O pipeline usa critérios objetivos para decidir:

**Qual clusterizador modelou melhor o dataset?**

Com base em:

- métricas de qualidade  
- quantidade de ruido  
- alinhamento entre métodos  
- estrutura dos clusters  

---

# **7. Visualizing the Results**

O pipeline salva várias imagens que você deve abrir e interpretar em sequência.

---

## **7.1 clusters_kmeans.png**

Perguntas:

- Os clusters estão bem separados?  
- Há regiões misturadas?  
- KMeans está forçando agrupamentos artificiais?  

---

## **7.2 clusters_hdbscan.png**

Perguntas:

- Quantos pontos noise (-1)?  
- Os clusters têm tamanhos coerentes?  
- Existem clusters pequenos porém significativos?  

Clusters pequenos porém densos → **cenários específicos**.  
Noise → **casos extremos**.

---

## **7.3 clusters_overlay.png**

Serve para ver:

- onde ambos concordam  
- onde discordam  
- como os clusters se sobrepõem  
- se HDBSCAN detecta subgrupos que KMeans não detecta  

Marcadores:

- círculo = KMeans  
- X = HDBSCAN  
- diamante preto = noise  

---

## **7.4 cluster_heatmap.png**

Matriz KMeans × HDBSCAN.

Interpretar diagonal:

- diagonal forte → ambos métodos convergem  
- diagonal fraca → estrutura complexa  
- noise forte → dataset difícil  

---

# **8. HTML Report (report.html)**

O relatório HTML consolida tudo:

- estatísticas  
- gráficos  
- tabelas  
- clusterizadores  
- métricas  
- heatmap  
- notas finais  

Serve como:

- documento de auditoria  
- análise de dataset  
- referência para ML  
- documentação do projeto  

---

# **9. How to Analyze a Dataset Using This Pipeline**

Aqui está a sequência ideal:

---

## **Passo 1 — Comece pelo relatório HTML**

Abra `report.html`.

Procure:

- número de clusters  
- ruido HDBSCAN  
- duplicatas  
- similaridade  
- outliers  

---

## **Passo 2 — Examine clusters individualmente**

Abra:

- `clusters_kmeans.png`  
- `clusters_hdbscan.png`  

Veja se:

- clusters fazem sentido visualmente  
- HDBSCAN encontra subgrupos relevantes  
- KMeans força agrupamentos errados  

---

## **Passo 3 — Cheque o overlay**

Se no overlay:

- círculos e X sobrepostos → métodos concordam  
- X agrupados onde círculos não → HDBSCAN encontra detalhes  
- diamantes pretos → imagens problemáticas  

---

## **Passo 4 — Analise o heatmap**

- diagonal forte = clusters robustos  
- off-diagonal = clusters ambíguos  
- coluna do noise = pontos esquisitos  

---

## **Passo 5 — Avaliação Automática**

Veja no painel:

- quem ganhou (KMeans ou HDBSCAN)  
- por que  
- métricas  

Se HDBSCAN ganhou → dataset tem estrutura complexa.  
Se KMeans ganhou → dataset mais uniforme.  

---

## **Passo 6 — Similaridade & Duplicação**

- alta duplicação → dataset redundante  
- muita similaridade → risco de overfitting  
- pouca similaridade → dataset diverso  

---

## **Passo 7 — Outliers**

Outliers são:

- candidatos a revisão  
- possíveis erros de captura  
- casos extremos que derrubam modelos ML  

---

# **10. Conclusion**

Este pipeline fornece:

- análise quantitativa  
- análise visual  
- explicações automáticas  
- documentação completa  
- limpeza e deduplicação  
- clusterização dupla  
- decisão automatizada entre algoritmos  

É uma ferramenta profissional para:

- curadoria de datasets  
- QA visual  
- preparo para deep learning  
- documentação técnica  